<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/enzo%2Fp10-8-clinical-expansion-preflight/74_P10_8_geometry_measurement_protocol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 74 — P10.8: protocolo geométrico para altura discal y diámetro AP

Este notebook **no realiza mediciones automáticas sobre pacientes**. Define y verifica
únicamente el contrato geométrico necesario para que, en una etapa posterior, un
profesional pueda seleccionar/revisar landmarks y el sistema convierta distancias de
píxeles a milímetros usando el `PixelSpacing` DICOM.

Alcance habilitado por el Notebook 73:

- `disc_height` — plano sagital;
- `ap_diameter` — plano axial.

Reglas de seguridad y alcance:

- no entrena;
- no carga ni deserializa `.pt`;
- no abre tests sellados;
- no crea ground truth clínico;
- no infiere landmarks automáticamente;
- no congela umbrales clínicos;
- no clasifica normalidad/severidad;
- requiere revisión profesional;
- no constituye diagnóstico clínico.


In [1]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Entorno no Colab")


Mounted at /content/drive


In [2]:
from __future__ import annotations

import hashlib
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

ROOT = Path(
    os.getenv(
        "PFI_ROOT",
        "/content/drive/MyDrive/PFI_MVP",
    )
)
PREF = Path(
    os.getenv(
        "PFI_P10_8_PREFLIGHT_ROOT",
        str(
            ROOT
            / "results"
            / "P10_8_clinical_expansion_preflight"
        ),
    )
)
N73_ROOT = Path(
    os.getenv(
        "PFI_P10_8_NOTEBOOK73_ROOT",
        str(PREF / "viability_gate"),
    )
)
OUT = Path(
    os.getenv(
        "PFI_P10_8_NOTEBOOK74_ROOT",
        str(PREF / "geometry_measurement_protocol"),
    )
)

MARKER_73 = N73_ROOT / "NOTEBOOK_73_COMPLETE.json"
CANDIDATES_73 = (
    N73_ROOT
    / "measurement_protocol_candidates_v1.csv"
)

if not MARKER_73.is_file():
    raise FileNotFoundError(
        f"Falta marcador Notebook 73: {MARKER_73}"
    )

marker_73 = json.loads(
    MARKER_73.read_text(encoding="utf-8")
)

if marker_73.get("status") != "NOTEBOOK_73_COMPLETE":
    raise RuntimeError(
        "Notebook 73 no está formalmente cerrado."
    )

if marker_73.get("schemaVersion") != (
    "pfi.p10-8.notebook-73-complete.v1"
):
    raise RuntimeError(
        "El marcador 73 no tiene el schemaVersion "
        "corregido esperado."
    )

if marker_73.get("trainingExecuted") is not False:
    raise RuntimeError(
        "Notebook 73 no declara trainingExecuted=false"
    )

if marker_73.get("weightsDeserialized") is not False:
    raise RuntimeError(
        "Notebook 73 no declara weightsDeserialized=false"
    )

if marker_73.get("trainingAuthorized") is not False:
    raise RuntimeError(
        "Notebook 73 no mantiene trainingAuthorized=false"
    )

if marker_73.get("outputPtFileCount") != 0:
    raise RuntimeError(
        "Notebook 73 reporta archivos .pt en su salida."
    )

if not CANDIDATES_73.is_file():
    raise FileNotFoundError(
        f"Falta salida de candidatos del Notebook 73: "
        f"{CANDIDATES_73}"
    )

candidates_73 = pd.read_csv(CANDIDATES_73)

required_findings = {"disc_height", "ap_diameter"}
observed_findings = set(
    candidates_73["findingType"]
    .dropna()
    .astype(str)
)

if observed_findings != required_findings:
    raise RuntimeError(
        "Notebook 74 esperaba exactamente "
        f"{sorted(required_findings)}, pero recibió "
        f"{sorted(observed_findings)}"
    )

protocol_allowed = (
    candidates_73["protocolDesignAllowed"]
    .astype(str)
    .str.lower()
    .isin({"true", "1", "yes"})
)
training_authorized = (
    candidates_73["trainingAuthorized"]
    .astype(str)
    .str.lower()
    .isin({"true", "1", "yes"})
)

if not protocol_allowed.all():
    raise RuntimeError(
        "Notebook 73 no habilitó ambos protocolos."
    )

if training_authorized.any():
    raise RuntimeError(
        "Existe una autorización de entrenamiento "
        "inesperada."
    )

print("Notebook 73 verificado.")
print("Protocolos habilitados:", sorted(observed_findings))
print("Salida Notebook 74:", OUT)


Notebook 73 verificado.
Protocolos habilitados: ['ap_diameter', 'disc_height']
Salida Notebook 74: /content/drive/MyDrive/PFI_MVP/results/P10_8_clinical_expansion_preflight/geometry_measurement_protocol


In [3]:
def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
    os.replace(temp, path)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()

def positive_finite(value: float, name: str) -> float:
    value = float(value)
    if not math.isfinite(value) or value <= 0:
        raise ValueError(
            f"{name} debe ser finito y > 0"
        )
    return value

def point_xy(point: tuple[float, float]) -> tuple[float, float]:
    if len(point) != 2:
        raise ValueError(
            "Cada landmark debe tener exactamente x,y"
        )
    x, y = map(float, point)
    if not (math.isfinite(x) and math.isfinite(y)):
        raise ValueError(
            "Las coordenadas deben ser finitas"
        )
    return x, y

def distance_mm(
    point_a: tuple[float, float],
    point_b: tuple[float, float],
    row_spacing_mm: float,
    column_spacing_mm: float,
) -> float:
    x1, y1 = point_xy(point_a)
    x2, y2 = point_xy(point_b)
    row_spacing_mm = positive_finite(
        row_spacing_mm,
        "row_spacing_mm",
    )
    column_spacing_mm = positive_finite(
        column_spacing_mm,
        "column_spacing_mm",
    )

    dx_mm = (x2 - x1) * column_spacing_mm
    dy_mm = (y2 - y1) * row_spacing_mm

    return math.hypot(dx_mm, dy_mm)

input_sha = sha256_file(CANDIDATES_73)

print("SHA256 measurement_protocol_candidates_v1.csv:")
print(input_sha)


SHA256 measurement_protocol_candidates_v1.csv:
768bf4d0f09b5eb8139ac743c966785070cdf226074ecf754862542f27621c66


## Definición geométrica conservadora

### Altura discal

Se definen **tres pares de landmarks revisados por un profesional** en una imagen
sagital seleccionada:

- anterior superior ↔ anterior inferior;
- medio superior ↔ medio inferior;
- posterior superior ↔ posterior inferior.

Cada par produce una distancia en milímetros. El notebook puede informar las tres
alturas y un promedio aritmético descriptivo. **No interpreta el promedio como normal,
disminuido ni patológico.**

### Diámetro AP

Se definen **dos landmarks revisados por un profesional** sobre una imagen axial
seleccionada:

- límite anterior del ROI anatómico que se quiera medir;
- límite posterior del mismo ROI.

La distancia física entre ambos landmarks es el diámetro AP. El contrato no fija
todavía qué borde anatómico debe generar automáticamente cada punto y no aplica
umbrales de severidad.


In [4]:
protocol_registry = pd.DataFrame(
    [
        {
            "measurementType": "disc_height",
            "plane": "sagittal",
            "measurementUnit": "mm",
            "landmarkSource":
                "professional_selected_or_reviewed",
            "automaticLandmarkDetectionValidated": False,
            "automaticMeasurementValidated": False,
            "clinicalThresholdFrozen": False,
            "severityClassificationAllowed": False,
            "professionalReviewRequired": True,
            "notClinicalDiagnosis": True,
            "trainingAuthorized": False,
            "outputFields":
                "anteriorHeightMm|middleHeightMm|"
                "posteriorHeightMm|meanHeightMm",
        },
        {
            "measurementType": "ap_diameter",
            "plane": "axial",
            "measurementUnit": "mm",
            "landmarkSource":
                "professional_selected_or_reviewed",
            "automaticLandmarkDetectionValidated": False,
            "automaticMeasurementValidated": False,
            "clinicalThresholdFrozen": False,
            "severityClassificationAllowed": False,
            "professionalReviewRequired": True,
            "notClinicalDiagnosis": True,
            "trainingAuthorized": False,
            "outputFields": "apDiameterMm",
        },
    ]
)

display(protocol_registry)


,measurementType,plane,measurementUnit,landmarkSource,automaticLandmarkDetectionValidated,automaticMeasurementValidated,clinicalThresholdFrozen,severityClassificationAllowed,professionalReviewRequired,notClinicalDiagnosis,trainingAuthorized,outputFields
0,disc_height,sagittal,mm,professional_selected_or_reviewed,False,False,False,False,True,True,False,anteriorHeightMm|middleHeightMm|posteriorHeigh...
1,ap_diameter,axial,mm,professional_selected_or_reviewed,False,False,False,False,True,True,False,apDiameterMm


In [5]:
landmark_schema = pd.DataFrame(
    [
        {
            "measurementType": "disc_height",
            "landmarkName": "anterior_superior",
            "plane": "sagittal",
            "required": True,
            "semanticRole":
                "superior boundary at anterior sampling position",
        },
        {
            "measurementType": "disc_height",
            "landmarkName": "anterior_inferior",
            "plane": "sagittal",
            "required": True,
            "semanticRole":
                "inferior boundary at anterior sampling position",
        },
        {
            "measurementType": "disc_height",
            "landmarkName": "middle_superior",
            "plane": "sagittal",
            "required": True,
            "semanticRole":
                "superior boundary at middle sampling position",
        },
        {
            "measurementType": "disc_height",
            "landmarkName": "middle_inferior",
            "plane": "sagittal",
            "required": True,
            "semanticRole":
                "inferior boundary at middle sampling position",
        },
        {
            "measurementType": "disc_height",
            "landmarkName": "posterior_superior",
            "plane": "sagittal",
            "required": True,
            "semanticRole":
                "superior boundary at posterior sampling position",
        },
        {
            "measurementType": "disc_height",
            "landmarkName": "posterior_inferior",
            "plane": "sagittal",
            "required": True,
            "semanticRole":
                "inferior boundary at posterior sampling position",
        },
        {
            "measurementType": "ap_diameter",
            "landmarkName": "anterior_boundary",
            "plane": "axial",
            "required": True,
            "semanticRole":
                "professional-reviewed anterior ROI boundary",
        },
        {
            "measurementType": "ap_diameter",
            "landmarkName": "posterior_boundary",
            "plane": "axial",
            "required": True,
            "semanticRole":
                "professional-reviewed posterior ROI boundary",
        },
    ]
)

landmark_schema["coordinateSystem"] = "pixel_xy"
landmark_schema["physicalCalibration"] = (
    "DICOM_PixelSpacing_row_column"
)
landmark_schema["automaticLocalizationValidated"] = False

display(landmark_schema)


,measurementType,landmarkName,plane,required,semanticRole,coordinateSystem,physicalCalibration,automaticLocalizationValidated
0,disc_height,anterior_superior,sagittal,True,superior boundary at anterior sampling position,pixel_xy,DICOM_PixelSpacing_row_column,False
1,disc_height,anterior_inferior,sagittal,True,inferior boundary at anterior sampling position,pixel_xy,DICOM_PixelSpacing_row_column,False
2,disc_height,middle_superior,sagittal,True,superior boundary at middle sampling position,pixel_xy,DICOM_PixelSpacing_row_column,False
3,disc_height,middle_inferior,sagittal,True,inferior boundary at middle sampling position,pixel_xy,DICOM_PixelSpacing_row_column,False
4,disc_height,posterior_superior,sagittal,True,superior boundary at posterior sampling position,pixel_xy,DICOM_PixelSpacing_row_column,False
5,disc_height,posterior_inferior,sagittal,True,inferior boundary at posterior sampling position,pixel_xy,DICOM_PixelSpacing_row_column,False
6,ap_diameter,anterior_boundary,axial,True,professional-reviewed anterior ROI boundary,pixel_xy,DICOM_PixelSpacing_row_column,False
7,ap_diameter,posterior_boundary,axial,True,professional-reviewed posterior ROI boundary,pixel_xy,DICOM_PixelSpacing_row_column,False


In [6]:
formula_registry = pd.DataFrame(
    [
        {
            "formulaId": "distance_mm_v1",
            "measurementType":
                "disc_height|ap_diameter",
            "formula":
                "sqrt(((x2-x1)*columnSpacingMm)^2 + "
                "((y2-y1)*rowSpacingMm)^2)",
            "requiresPixelSpacing": True,
            "supportsAnisotropicPixels": True,
            "clinicalInterpretation": "none",
        },
        {
            "formulaId": "disc_height_mean_v1",
            "measurementType": "disc_height",
            "formula":
                "(anteriorHeightMm + middleHeightMm + "
                "posteriorHeightMm) / 3",
            "requiresPixelSpacing": True,
            "supportsAnisotropicPixels": True,
            "clinicalInterpretation":
                "descriptive_only_no_threshold",
        },
    ]
)

display(formula_registry)


,formulaId,measurementType,formula,requiresPixelSpacing,supportsAnisotropicPixels,clinicalInterpretation
0,distance_mm_v1,disc_height|ap_diameter,sqrt(((x2-x1)*columnSpacingMm)^2 + ((y2-y1)*ro...,True,True,none
1,disc_height_mean_v1,disc_height,(anteriorHeightMm + middleHeightMm + posterior...,True,True,descriptive_only_no_threshold


In [7]:
review_requirements = pd.DataFrame(
    [
        {
            "ruleId": "R01",
            "measurementType": "disc_height",
            "requirement":
                "Sagittal slice must be selected/reviewed by "
                "a professional.",
            "blocking": True,
        },
        {
            "ruleId": "R02",
            "measurementType": "disc_height",
            "requirement":
                "All six landmarks must be visible and "
                "professionally reviewed.",
            "blocking": True,
        },
        {
            "ruleId": "R03",
            "measurementType": "disc_height",
            "requirement":
                "PixelSpacing must be present and positive.",
            "blocking": True,
        },
        {
            "ruleId": "R04",
            "measurementType": "ap_diameter",
            "requirement":
                "Axial slice and anatomical ROI must be "
                "selected/reviewed by a professional.",
            "blocking": True,
        },
        {
            "ruleId": "R05",
            "measurementType": "ap_diameter",
            "requirement":
                "Anterior and posterior boundary landmarks "
                "must be professionally reviewed.",
            "blocking": True,
        },
        {
            "ruleId": "R06",
            "measurementType": "ap_diameter",
            "requirement":
                "PixelSpacing must be present and positive.",
            "blocking": True,
        },
        {
            "ruleId": "R07",
            "measurementType": "all",
            "requirement":
                "No severity label or clinical threshold may "
                "be produced by this protocol.",
            "blocking": True,
        },
        {
            "ruleId": "R08",
            "measurementType": "all",
            "requirement":
                "Original image must remain available for "
                "professional review.",
            "blocking": True,
        },
    ]
)

display(review_requirements)


,ruleId,measurementType,requirement,blocking
0,R01,disc_height,Sagittal slice must be selected/reviewed by a ...,True
1,R02,disc_height,All six landmarks must be visible and professi...,True
2,R03,disc_height,PixelSpacing must be present and positive.,True
3,R04,ap_diameter,Axial slice and anatomical ROI must be selecte...,True
4,R05,ap_diameter,Anterior and posterior boundary landmarks must...,True
5,R06,ap_diameter,PixelSpacing must be present and positive.,True
6,R07,all,No severity label or clinical threshold may be...,True
7,R08,all,Original image must remain available for profe...,True


In [8]:
# Pruebas geométricas sintéticas. No usan imágenes ni datos de pacientes.
synthetic_cases = [
    {
        "testId": "iso_horizontal",
        "pointA": (10.0, 20.0),
        "pointB": (14.0, 20.0),
        "rowSpacingMm": 0.5,
        "columnSpacingMm": 0.5,
        "expectedMm": 2.0,
    },
    {
        "testId": "iso_vertical",
        "pointA": (10.0, 20.0),
        "pointB": (10.0, 26.0),
        "rowSpacingMm": 0.5,
        "columnSpacingMm": 0.5,
        "expectedMm": 3.0,
    },
    {
        "testId": "anisotropic_diagonal",
        "pointA": (0.0, 0.0),
        "pointB": (3.0, 4.0),
        "rowSpacingMm": 0.5,
        "columnSpacingMm": 1.0,
        "expectedMm": math.sqrt(13.0),
    },
]

synthetic_rows = []

for case in synthetic_cases:
    measured = distance_mm(
        case["pointA"],
        case["pointB"],
        case["rowSpacingMm"],
        case["columnSpacingMm"],
    )
    absolute_error = abs(
        measured - case["expectedMm"]
    )

    synthetic_rows.append(
        {
            "testId": case["testId"],
            "measuredMm": measured,
            "expectedMm": case["expectedMm"],
            "absoluteErrorMm": absolute_error,
            "passed": absolute_error < 1e-12,
            "patientDataUsed": False,
        }
    )

synthetic_tests = pd.DataFrame(synthetic_rows)

if not synthetic_tests["passed"].all():
    raise RuntimeError(
        "Falló una prueba sintética del contrato geométrico."
    )

display(synthetic_tests)


,testId,measuredMm,expectedMm,absoluteErrorMm,passed,patientDataUsed
0,iso_horizontal,2.000000,2.000000,0.0,True,False
1,iso_vertical,3.000000,3.000000,0.0,True,False
2,anisotropic_diagonal,3.605551,3.605551,0.0,True,False


In [9]:
measurement_contract = {
    "schemaVersion":
        "pfi.p10-8.geometry-measurement-protocol.v1",
    "notClinicalDiagnosis": True,
    "professionalReviewRequired": True,
    "automaticLandmarkDetectionValidated": False,
    "automaticMeasurementValidated": False,
    "clinicalThresholdsFrozen": False,
    "severityClassificationAllowed": False,
    "trainingAuthorized": False,
    "coordinateSystem": "pixel_xy",
    "physicalCalibration": {
        "source": "DICOM PixelSpacing",
        "rowSpacingField": "rowSpacingMm",
        "columnSpacingField": "columnSpacingMm",
        "anisotropicPixelsSupported": True,
    },
    "measurements": {
        "discHeight": {
            "plane": "sagittal",
            "requiredLandmarks": [
                "anterior_superior",
                "anterior_inferior",
                "middle_superior",
                "middle_inferior",
                "posterior_superior",
                "posterior_inferior",
            ],
            "outputs": [
                "anteriorHeightMm",
                "middleHeightMm",
                "posteriorHeightMm",
                "meanHeightMm",
            ],
            "clinicalInterpretation": None,
        },
        "apDiameter": {
            "plane": "axial",
            "requiredLandmarks": [
                "anterior_boundary",
                "posterior_boundary",
            ],
            "outputs": ["apDiameterMm"],
            "clinicalInterpretation": None,
        },
    },
}

print(
    json.dumps(
        measurement_contract,
        indent=2,
        ensure_ascii=False,
    )
)


{
  "schemaVersion": "pfi.p10-8.geometry-measurement-protocol.v1",
  "notClinicalDiagnosis": true,
  "professionalReviewRequired": true,
  "automaticLandmarkDetectionValidated": false,
  "automaticMeasurementValidated": false,
  "clinicalThresholdsFrozen": false,
  "severityClassificationAllowed": false,
  "trainingAuthorized": false,
  "coordinateSystem": "pixel_xy",
  "physicalCalibration": {
    "source": "DICOM PixelSpacing",
    "rowSpacingField": "rowSpacingMm",
    "columnSpacingField": "columnSpacingMm",
    "anisotropicPixelsSupported": true
  },
  "measurements": {
    "discHeight": {
      "plane": "sagittal",
      "requiredLandmarks": [
        "anterior_superior",
        "anterior_inferior",
        "middle_superior",
        "middle_inferior",
        "posterior_superior",
        "posterior_inferior"
      ],
      "outputs": [
        "anteriorHeightMm",
        "middleHeightMm",
        "posteriorHeightMm",
        "meanHeightMm"
      ],
      "clinicalInterpretatio

In [10]:
OUT.mkdir(parents=True, exist_ok=True)

output_paths = {
    "protocolRegistry":
        OUT / "measurement_protocol_registry_v1.csv",
    "landmarkSchema":
        OUT / "measurement_landmark_schema_v1.csv",
    "formulaRegistry":
        OUT / "measurement_formula_registry_v1.csv",
    "reviewRequirements":
        OUT / "professional_review_requirements_v1.csv",
    "syntheticTests":
        OUT / "synthetic_geometry_tests_v1.csv",
    "contract":
        OUT / "geometry_measurement_contract_v1.json",
    "inputHashes":
        OUT / "notebook73_input_hashes_v1.csv",
    "summary":
        OUT / "NOTEBOOK_74_SUMMARY.json",
}

protocol_registry.to_csv(
    output_paths["protocolRegistry"],
    index=False,
)
landmark_schema.to_csv(
    output_paths["landmarkSchema"],
    index=False,
)
formula_registry.to_csv(
    output_paths["formulaRegistry"],
    index=False,
)
review_requirements.to_csv(
    output_paths["reviewRequirements"],
    index=False,
)
synthetic_tests.to_csv(
    output_paths["syntheticTests"],
    index=False,
)
write_json(
    output_paths["contract"],
    measurement_contract,
)
pd.DataFrame(
    [
        {
            "inputName":
                "measurement_protocol_candidates_v1.csv",
            "sha256": input_sha,
        }
    ]
).to_csv(
    output_paths["inputHashes"],
    index=False,
)

pt_files = list(OUT.rglob("*.pt"))

if pt_files:
    raise RuntimeError(
        "La salida del Notebook 74 contiene .pt inesperados."
    )

summary = {
    "schemaVersion":
        "pfi.p10-8.notebook-74-summary.v1",
    "generatedAtUtc":
        datetime.now(timezone.utc).isoformat(),
    "trainingExecuted": False,
    "weightsDeserialized": False,
    "internalTestAccessed": False,
    "officialHiddenTestAccessed": False,
    "patientIdentifiersExported": False,
    "clinicalGroundTruthCreated": False,
    "clinicalThresholdsFrozen": False,
    "automaticLandmarkDetectionValidated": False,
    "automaticMeasurementValidated": False,
    "trainingAuthorized": False,
    "notClinicalDiagnosis": True,
    "professionalReviewRequired": True,
    "protocolCount":
        int(len(protocol_registry)),
    "measurementTypes": [
        "disc_height",
        "ap_diameter",
    ],
    "landmarkDefinitionCount":
        int(len(landmark_schema)),
    "formulaCount":
        int(len(formula_registry)),
    "reviewRuleCount":
        int(len(review_requirements)),
    "syntheticTestCount":
        int(len(synthetic_tests)),
    "syntheticTestsPassed":
        bool(synthetic_tests["passed"].all()),
    "outputPtFileCount": len(pt_files),
    "nextRequiredGate":
        "NOTEBOOK_75_MULTIFRAME_HERNIA_REVIEW_ONLY_NO_RETRAINING",
}

write_json(
    output_paths["summary"],
    summary,
)

marker = {
    **summary,
    "schemaVersion":
        "pfi.p10-8.notebook-74-complete.v1",
    "summarySchemaVersion":
        summary["schemaVersion"],
    "status": "NOTEBOOK_74_COMPLETE",
    "outputs": {
        key: str(value)
        for key, value in output_paths.items()
    },
}

write_json(
    OUT / "NOTEBOOK_74_COMPLETE.json",
    marker,
)

print(
    json.dumps(
        marker,
        indent=2,
        ensure_ascii=False,
    )
)
print("NOTEBOOK_74_COMPLETE")


{
  "schemaVersion": "pfi.p10-8.notebook-74-complete.v1",
  "generatedAtUtc": "2026-08-07T02:46:18.817887+00:00",
  "trainingExecuted": false,
  "weightsDeserialized": false,
  "internalTestAccessed": false,
  "officialHiddenTestAccessed": false,
  "patientIdentifiersExported": false,
  "clinicalGroundTruthCreated": false,
  "clinicalThresholdsFrozen": false,
  "automaticLandmarkDetectionValidated": false,
  "automaticMeasurementValidated": false,
  "trainingAuthorized": false,
  "notClinicalDiagnosis": true,
  "professionalReviewRequired": true,
  "protocolCount": 2,
  "measurementTypes": [
    "disc_height",
    "ap_diameter"
  ],
  "landmarkDefinitionCount": 8,
  "formulaCount": 2,
  "reviewRuleCount": 8,
  "syntheticTestCount": 3,
  "syntheticTestsPassed": true,
  "outputPtFileCount": 0,
  "nextRequiredGate": "NOTEBOOK_75_MULTIFRAME_HERNIA_REVIEW_ONLY_NO_RETRAINING",
  "summarySchemaVersion": "pfi.p10-8.notebook-74-summary.v1",
  "status": "NOTEBOOK_74_COMPLETE",
  "outputs": {
   

## Interpretación obligatoria

Este notebook valida **el contrato matemático y de revisión**, no la precisión clínica
de una medición automática.

Hasta que exista una etapa posterior de localización anatómica validada:

- los landmarks deben ser seleccionados o revisados por un profesional;
- no se deben generar categorías de severidad;
- no se deben aplicar umbrales diagnósticos;
- los valores en milímetros son mediciones geométricas descriptivas;
- el resultado debe presentarse como apoyo a la revisión profesional.
